# 01 - Data Cleaning

Load the raw transaction data and inspect the effect of the cleaning
steps defined in `src/data/preprocess.py`.

In [1]:
import os
import sys

# Move up to the project root so relative paths in config.yaml resolve correctly.
# Guard against running this cell twice (which would try to go up an extra level).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.append(os.getcwd())

from src.data.load_data import load_config, load_raw_data
from src.data.preprocess import clean_transactions

config = load_config("config.yaml")
raw = load_raw_data(config)
print("Raw shape:", raw.shape)
raw.head()

Raw shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/01/2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/01/2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/01/2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/01/2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/01/2010 08:26,3.39,17850.0,United Kingdom


## Apply cleaning

Removes missing CustomerIDs, cancelled orders, and non-positive
quantity/price rows; derives `TotalPrice`.

In [2]:
clean = clean_transactions(raw)
print("Clean shape:", clean.shape)
print(f"Rows removed: {raw.shape[0] - clean.shape[0]}")
clean.head()

Clean shape: (397884, 9)
Rows removed: 144025


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [3]:
# Quick look at what got removed
missing_ids = raw["CustomerID"].isnull().sum()
cancelled = raw["InvoiceNo"].astype(str).str.startswith("C").sum()
non_positive = ((raw["Quantity"] <= 0) | (raw["UnitPrice"] <= 0)).sum()

print(f"Missing CustomerID: {missing_ids}")
print(f"Cancelled orders: {cancelled}")
print(f"Non-positive quantity/price: {non_positive}")

Missing CustomerID: 135080
Cancelled orders: 9288
Non-positive quantity/price: 11805
